In [1]:
import pandas as pd
import folium
import webbrowser
from datetime import datetime
import os
from folium.plugins import BoatMarker

from boats.lib.common import DIR_HOME, timestamp
from boats.lib.boat import Boat
from boats.lib.common import MY_USER, get_boat_race_data, DIR_HTML

In [2]:
file_timestamp = timestamp().replace(':', '_').replace('/', ' ').replace(' ', '_')

In [3]:
zoom_start = 8

In [4]:
fname = 'MARK 3'
f_csv_exported   = os.path.join(DIR_HOME, 'routes', 'exported', '{}.csv'.format(fname))
f_csv_fixed = os.path.join(DIR_HOME, 'routes', 'exported', '{}_fixed.csv'.format(fname))
f_json  = os.path.join(DIR_HOME, 'routes', 'exported', '{}.json'.format(fname))
f_route = os.path.join(DIR_HOME, 'routes', 'import', 'petsamo_{}.txt'.format(file_timestamp))

In [5]:
with open(f_csv_exported, 'r') as f:
    contents = f.read()

In [6]:
with open(f_csv_fixed, 'w') as f:
    f.write(contents.replace(';', ';,'))

In [7]:
df=pd.read_csv(f_csv_fixed)

In [8]:
df.drop(0, axis=0, inplace=True)
df

,position;,heure
1,01°40.546 S 131°13.184 E;,28/01/2023 09:05:00
2,01°34.809 S 130°55.710 E;,28/01/2023 13:35:00
3,01°18.397 S 130°33.903 E;,28/01/2023 17:30:00
4,00°51.736 S 130°19.684 E;,28/01/2023 21:45:00
5,00°27.702 S 130°03.901 E;,29/01/2023 02:25:00
6,00°11.477 N 129°51.049 E;,29/01/2023 09:35:00
7,02°39.627 N 129°52.867 E;,30/01/2023 05:25:00
8,04°06.543 N 129°39.871 E;,30/01/2023 15:25:00
9,12°07.925 N 125°49.430 E;,01/02/2023 16:20:00
10,12°44.689 N 124°59.121 E;,01/02/2023 21:20:00


In [9]:
text = '\n'.join(list(df['position;']))

In [11]:
with open(f_route, 'w') as f:
          f.write(text)

In [12]:
df_json = pd.read_json(f_json)
df_json.head()

,nom,tracks
0,MARK 3,"[1674845912, 131436.0683333333, -3063.2866666667]"
1,MARK 3,"[1674897000, 131219.7328527376, -1675.7647599179]"
2,MARK 3,"[1674912900, 130928.4920214960, -1580.1547781008]"
3,MARK 3,"[1674926700, 130565.0447120234, -1306.6146376754]"
4,MARK 3,"[1674942000, 130328.0740687841, -862.2690776211]"


In [13]:
df_json.drop('nom', axis=1, inplace=True)
df_json.head()
df_json.drop(0, axis=0, inplace=True)

In [14]:
df_json['epoch'] = [df_json['tracks'][idx][0] for idx in df_json.index]
df_json['Lon'] = [float(df_json['tracks'][idx][1])/1000 for idx in df_json.index]
df_json['Lat'] = [float(df_json['tracks'][idx][2])/1000 for idx in df_json.index]
df_json.drop('tracks', axis=1, inplace=True)
df_json.head()

,epoch,Lon,Lat
1,1674897000,131.219733,-1.675765
2,1674912900,130.928492,-1.580155
3,1674926700,130.565045,-1.306615
4,1674942000,130.328074,-0.862269
5,1674958800,130.065022,-0.461697


In [15]:
df_json['epoch']=df_json['epoch'].astype(int)

In [16]:
df_json['ETA'] = [datetime.fromtimestamp(x).strftime("%d-%h %H:%M") for x in df_json['epoch']]
df_json.drop('epoch', axis=1, inplace=True)

In [17]:
df_json

,Lon,Lat,ETA
1,131.219733,-1.675765,28-Jan 04:10
2,130.928492,-1.580155,28-Jan 08:35
3,130.565045,-1.306615,28-Jan 12:25
4,130.328074,-0.862269,28-Jan 16:40
5,130.065022,-0.461697,28-Jan 21:20
6,129.850825,0.191276,29-Jan 04:30
7,129.881112,2.660445,30-Jan 00:20
8,129.664523,4.109042,30-Jan 10:20
9,125.823841,12.132081,01-Feb 11:15
10,124.985353,12.744821,01-Feb 16:20


In [18]:
points = [(df_json.iloc[i]['Lat'], df_json.iloc[i]['Lon']) for i in range(len(df_json.index))]

In [19]:
mymap = folium.Map(location=[df_json.iloc[1]['Lat'], df_json.iloc[1]['Lon']], zoom_start=zoom_start)

In [20]:
boat_name = 'Petsamo'
race = 'Stardust'
user = 'Viper Vit'
oBoat = Boat(boat_name)
oBoat.getdata()
curr_pos = [round(oBoat.pos[0], 3), round(oBoat.pos[1], 3)]
sog = oBoat.nav['sog']
hdg = oBoat.nav['hdg']
race_data = get_boat_race_data(race, user, boat_name)
rank = race_data['rank']
track = race_data['track']
track.reverse()
track.append(curr_pos)

df_track = pd.DataFrame(track)
df_track.columns = ['Lat', 'Lon']

In [21]:
folium.PolyLine(points, color='red').add_to(mymap)
folium.PolyLine(track).add_to(mymap)
BoatMarker(curr_pos, color='blue',
           heading=oBoat.nav['hdg'],
           wind_heading=oBoat.wind['twd'],
           wind_speed=oBoat.wind['tws']).add_to(mymap)
mymap